# 🥚 Egg Detection System using Deep Learning and YOLOv11n

Detect and count eggs in images, videos and live camera streams with a **YOLOv11n** (CNN-based) object detector fine-tuned on a Kaggle dataset.

**Run the cells from top to bottom.** Sections: setup → dataset → training → evaluation → prediction.

## 1. Install required packages

In [ ]:
%pip install -q -r ../requirements.txt

## 2. Imports and configuration

In [ ]:
import sys
from pathlib import Path

# make the project's src/ folder importable
SRC_DIR = Path.cwd().parent / "src" if Path.cwd().name == "notebooks" else Path.cwd() / "src"
sys.path.insert(0, str(SRC_DIR))

import cv2
import matplotlib.pyplot as plt

import utils, train, evaluate, predict

# ---------------- user settings ----------------
KAGGLE_DATASET = utils.KAGGLE_DATASET   # or write e.g. "owner/egg-detection"
EPOCHS = 50
IMG_SIZE = 640
BATCH_SIZE = 16
LEARNING_RATE = 0.01
PATIENCE = 15                            # early stopping
CONFIDENCE_THRESHOLD = 0.5               # only detections >= this are displayed
EGG_CLASS_IDS = None                     # None = every annotated object is an egg
# -----------------------------------------------

DEVICE = utils.get_device()              # GPU if CUDA is available, otherwise CPU
print("Project root:", utils.PROJECT_ROOT)

## 3. Kaggle API setup (secure)

1. On kaggle.com open **Settings → API → Create New Token** to download `kaggle.json`.
2. Move it to `~/.kaggle/kaggle.json` and run `chmod 600 ~/.kaggle/kaggle.json` (Windows: `C:\Users\<you>\.kaggle\`).
3. Or set the environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY`.

> **Never** commit `kaggle.json` to GitHub or paste your key into a cell. The `.gitignore` already excludes it.

To discover a suitable dataset, run the optional search below and copy the `owner/slug` into `KAGGLE_DATASET` above.

In [ ]:
# Optional: list Kaggle datasets that match a search term (requires credentials)
# utils.search_kaggle("egg detection")

## 4. Download the dataset programmatically

In [ ]:
try:
    utils.download_kaggle_dataset(KAGGLE_DATASET)
except (FileNotFoundError, ValueError, ImportError) as err:
    print("[ERROR]", err)
    print("If you already have a YOLO-format egg dataset, copy it into", utils.RAW_DIR, "and continue.")

## 5. Prepare the YOLO-format dataset and create `data.yaml`

In [ ]:
counts = utils.prepare_dataset(utils.RAW_DIR, utils.DATASET_DIR, keep_class_ids=EGG_CLASS_IDS)
print(open(utils.DATA_YAML).read())

## 6. Check the dataset structure

In [ ]:
summary = utils.check_dataset_structure()

## 7. Sample images with their annotations

In [ ]:
utils.show_samples("train", n=6)

## 8. Train YOLOv11n

The pretrained `yolo11n.pt` weights (COCO) are fine-tuned on the egg dataset – this is **transfer learning**.

In [ ]:
best_weights, run_dir = train.train(
    epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH_SIZE,
    lr0=LEARNING_RATE, patience=PATIENCE,
)
print("Best weights:", best_weights)
print("Run folder  :", run_dir)

## 9. Training / validation loss curves

In [ ]:
history = utils.plot_training_curves(utils.RUNS_DIR / utils.RUN_NAME / "results.csv",
                                     save_path=utils.OUTPUTS_DIR / "training_curves.png")
history.tail(3)

## 10. Evaluate on the test set

In [ ]:
results = evaluate.evaluate(split="test", conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE)

## 11. Evaluation visualisations

In [ ]:
plots = Path(results["plots_dir"])
utils.show_images(
    [plots / "confusion_matrix.png", plots / "BoxPR_curve.png", plots / "BoxF1_curve.png", plots / "BoxP_curve.png"],
    ["Confusion matrix", "Precision-Recall curve", "F1-confidence curve", "Precision-confidence curve"],
)

## 12. Sample predictions on test images

In [ ]:
evaluate.save_sample_predictions(split="test", n=4, conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE, show=True)

## 13. Predict on a single image

In [ ]:
model = utils.load_model(utils.DEFAULT_WEIGHTS)
test_images = utils.list_images(utils.DATASET_DIR / "images" / "test")
SINGLE_IMAGE = test_images[0]            # or: Path("path/to/your_image.jpg")

n_eggs = predict.predict_image(model, SINGLE_IMAGE, conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE, device=DEVICE)

## 14. Predict on multiple images

In [ ]:
counts = predict.predict_images(model, test_images[:5], conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE, device=DEVICE)
counts

## 15. Predict on a video

In [ ]:
VIDEO_PATH = Path("../sample_video.mp4")  # put your own video here

if VIDEO_PATH.exists():
    out_video, frame_counts = predict.predict_video(model, VIDEO_PATH, conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE, device=DEVICE)
    plt.plot(frame_counts); plt.xlabel("frame"); plt.ylabel("eggs detected"); plt.title("Eggs per frame"); plt.show()
else:
    print(f"[INFO] No video at {VIDEO_PATH.resolve()} - set VIDEO_PATH to try this cell.")

## 16. Webcam / live camera

Opens an OpenCV window (press **q** to quit, **s** for a snapshot). It needs a local machine with a camera and display, so it is switched off by default.

In [ ]:
RUN_WEBCAM = False

if RUN_WEBCAM:
    try:
        predict.predict_webcam(model, camera=0, conf=CONFIDENCE_THRESHOLD, imgsz=IMG_SIZE, device=DEVICE)
    except RuntimeError as err:
        print("[ERROR]", err)

## 17. Summary

| Item | Value |
|---|---|
| Model | YOLOv11n (fine-tuned from COCO-pretrained weights) |
| Task | Single-class object detection (`0: egg`) |
| Outputs | `models/egg_yolo11n_best.pt`, `outputs/`, `runs/` |

Scroll back to the evaluation section for Precision, Recall, mAP@50 and mAP@50-95 on your dataset.